In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.stats import multivariate_normal
import cvxpy as cp
import torch
import normflows as nf
from torch.utils.data import TensorDataset, DataLoader, random_split
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GMM_COVARIANCE_TYPE = "diag"
GMM_REG_COVAR = 1e-3
GMM_N_INIT = 10

In [ ]:
######################### 2-Wass GMM DRO function ###################################
def Portfolio_2_Wass_MCVaR(xi, eps, tau, eta):
    N, d = xi.shape
    lda = cp.Variable(nonneg=True)
    s = cp.Variable(N)
    x = cp.Variable(d, nonneg=True)
    beta = cp.Variable()

    const = []
    for i in range(N):
        xi_norm2 = float(np.sum(xi[i]**2))
        const.append(cp.norm2(cp.hstack([2 * lda * xi[i] - ((1 / tau) + eta) * x, lda * xi_norm2 + s[i] + (beta/tau) - beta- lda]))
                     <= lda * xi_norm2 + s[i] + (beta/tau) - beta + lda)
        const.append(cp.norm2(cp.hstack([2 * lda * xi[i] - eta * x, lda * xi_norm2 + s[i] -  beta - lda]))
                     <= lda * xi_norm2 + s[i] - beta + lda)
        const.append(lda * xi_norm2 + s[i] >= -(beta/tau) + beta)
        const.append(lda * xi_norm2 + s[i] >= beta)
    const.append(cp.sum(x) == 1)

    obj = cp.Minimize(lda * (eps**2) + (1 / N) * cp.sum(s))
    prob = cp.Problem(obj, const)
    prob.solve(solver=cp.MOSEK)
    return x.value

def transforming_conditional(s, num_components, mu_k, sig_k, p_k, dim_s):
    reg = 1e-6
    eig_floor = 1e-10
    s = np.asarray(s).reshape(-1)
    mu_cond = []
    cov_cond = []
    log_weights = []
    for k in range(num_components):
        mu = np.asarray(mu_k[k])
        sigma = np.asarray(sig_k[k]).copy()
        mu_s = mu[:dim_s]
        mu_xi = mu[dim_s:]
        sigma_ss = sigma[:dim_s, :dim_s].copy()
        sigma_sx = sigma[:dim_s, dim_s:].copy()
        sigma_xs = sigma[dim_s:, :dim_s].copy()
        sigma_xx = sigma[dim_s:, dim_s:].copy()
        sigma_ss_reg = sigma_ss + reg * np.eye(dim_s)
        try:
            sigma_ss_inv = np.linalg.inv(sigma_ss_reg)
        except np.linalg.LinAlgError:
            sigma_ss_inv = np.linalg.pinv(sigma_ss_reg)
        cond_mu = mu_xi + sigma_xs @ sigma_ss_inv @ (s - mu_s)
        cond_cov = sigma_xx - sigma_xs @ sigma_ss_inv @ sigma_sx
        cond_cov = 0.5 * (cond_cov + cond_cov.T)
        eigvals = np.linalg.eigvalsh(cond_cov)
        min_eig = eigvals.min()
        if min_eig < eig_floor:
            cond_cov += (eig_floor - min_eig + reg) * np.eye(cond_cov.shape[0])
        try:
            log_weight = (np.log(max(p_k[k], 1e-300))+multivariate_normal.logpdf(s, mean=mu_s, cov=sigma_ss_reg, allow_singular=True))
        except Exception:
            log_weight = -np.inf
        mu_cond.append(cond_mu)
        cov_cond.append(cond_cov)
        log_weights.append(log_weight)
    log_weights = np.asarray(log_weights)
    if not np.any(np.isfinite(log_weights)):
        weights = np.ones(num_components) / num_components
    else:
        max_log_weight = np.max(log_weights[np.isfinite(log_weights)])
        weights = np.exp(log_weights - max_log_weight)
        weights[~np.isfinite(weights)] = 0.0
        if weights.sum() <= 1e-12:
            weights = np.ones(num_components) / num_components
        else:
            weights = weights / weights.sum()

    return np.array(mu_cond), np.array(cov_cond), weights

def MC_sampling(K, N, mu_list, cov_list, p_list, seed=None):
    d = mu_list.shape[1]
    samples = np.zeros((N, d))
    rng = np.random.default_rng(seed)
    for i in range(N):
        k = rng.choice(K, p=p_list)
        samples[i] = rng.multivariate_normal(mu_list[k], cov_list[k])
    return samples

def oos_loss_portfolio(x, xi, tau, eta):
    x = np.asarray(x, dtype=float).reshape(-1)
    xi = np.asarray(xi, dtype=float)

    if xi.ndim == 1:
        xi = xi.reshape(1, -1)

    loss = -xi @ x                  
    E_mean = -np.mean(loss)                  

    T = loss.shape[0]
    k = int(np.ceil(tau * T))
    k = max(1, k)

    worst_losses = np.partition(loss, -k)[-k:]
    cvar_loss = float(np.mean(worst_losses))

    return cvar_loss - eta * E_mean 

def oos_loss_valid(x, xi, tau, eta):
    return oos_loss_portfolio(x, xi, tau, eta)

def oos_mean_portfolio(x, xi):
    x = np.asarray(x).reshape(-1)
    xi = np.asarray(xi).reshape(-1)
    return float(x @ xi)

def select_K_by_AIC(z_np, max_K, seed=42, covariance_type=GMM_COVARIANCE_TYPE, reg_covar=GMM_REG_COVAR):
    aic_scores = []
    for k in range(1, max_K + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type=covariance_type,
            reg_covar=reg_covar,
            n_init=GMM_N_INIT,
            random_state=seed,
        )
        gmm.fit(z_np)
        aic_scores.append(gmm.aic(z_np))
    best_K = np.argmin(aic_scores) + 1
    return best_K

In [ ]:
def train_nf_model(latent_size, best_K, hidden_node, hidden_layer, num_bins, block_size, total_epoch, x, device, base_gmm, batch_size=64, lr=1e-3):
    patience = 30
    val_split = 0.2

    if base_gmm is None:
        raise ValueError("train_nf_model requires a pre-fitted base_gmm.")

    if base_gmm.covariance_type == "diag":
        scale = np.sqrt(base_gmm.covariances_)
    elif base_gmm.covariance_type == "full":
        diag_cov = np.array([np.diag(C) for C in base_gmm.covariances_])
        scale = np.sqrt(diag_cov)
    else:
        raise ValueError(f"Unsupported covariance_type: {base_gmm.covariance_type}")

    means = torch.tensor(base_gmm.means_, dtype=torch.float32, device=device)
    stds = torch.tensor(scale, dtype=torch.float32, device=device)
    weights = torch.tensor(base_gmm.weights_, dtype=torch.float32, device=device)

    flows = [nf.flows.AutoregressiveRationalQuadraticSpline(latent_size, hidden_layer, hidden_node, num_bins=num_bins) for _ in range(block_size)]

    q0 = nf.distributions.GaussianMixture(n_modes=best_K, dim=latent_size, loc=means, scale=stds, weights=weights, trainable=False)
    nfm = nf.NormalizingFlow(q0=q0, flows=flows).to(device)
    optimizer = torch.optim.Adam(nfm.parameters(), lr=lr)
    loss_hist = []

    N = x.size(0)
    val_size = int(N * val_split)
    train_size = N - val_size
    train_dataset, val_dataset = random_split(TensorDataset(x), [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=val_size, shuffle=False)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(total_epoch):
        nfm.train()
        train_loss_epoch = 0.0
        for batch in train_loader:
            x_batch = batch[0].to(device)
            optimizer.zero_grad()
            loss = nfm.forward_kld(x_batch)
            if not torch.isnan(loss):
                loss.backward()
                optimizer.step()
                train_loss_epoch += loss.item()
        loss_hist.append(train_loss_epoch)

        nfm.eval()
        with torch.no_grad():
            for val_batch in val_loader:
                x_val = val_batch[0].to(device)
                val_loss = nfm.forward_kld(x_val).item()

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_model_state = {k: v.detach().cpu().clone() for k, v in nfm.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}, best val loss: {best_val_loss:.4f}")
                break

    if best_model_state is not None:
        nfm.load_state_dict(best_model_state)

    return nfm, loss_hist

def inverse(nfm, x):
    with torch.no_grad():
        z_np = nfm.inverse(x).cpu().numpy()
    return z_np

def forward(nfm, z):
    with torch.no_grad():
        x = nfm.forward(z).cpu().numpy()
    return x

In [ ]:
def equal_weight_kernel(X_mat: np.array,Y_mat: np.array,X0: np.array) -> np.array:
    Y_mat = np.asarray(Y_mat, dtype=np.float64)
    num_assets = Y_mat.shape[1]
    return np.ones(num_assets) / num_assets

def mean_CVaR_kernel(X_mat:np.array, Y_mat:np.array, X0:np.array, reg_params:float, tau:float,)->np.array:
    Y_mat = np.asarray(Y_mat, dtype=np.float64)

    num_sample = Y_mat.shape[0]
    dim_beta = Y_mat.shape[1]
    alpha = cp.Variable(shape = (1,), name = 'alpha')
    beta = cp.Variable(shape = (dim_beta,), name = 'beta', nonneg=True)
    lambda_ = cp.Variable(shape = (num_sample,), name = 'lambda')
    constraints = [
        cp.sum(beta) == 1,
        lambda_ >= -reg_params*(Y_mat@beta) + alpha,
        lambda_ >= -(reg_params+1/tau)*(Y_mat@beta) + (1-1/tau)*alpha,
    ]
    problem = cp.Problem(cp.Minimize(cp.sum(lambda_)), constraints)
    problem.solve()
    if problem.status != 'optimal':
        raise ValueError('problem is not optimal')
    return beta.value

def DR_mean_CVaR_kernel(X_mat: np.array, Y_mat: np.array, X0: np.array, reg_params: float, tau: float, rho: float):
    Y_mat = np.asarray(Y_mat, dtype=np.float64)

    num_sample = Y_mat.shape[0]
    dim_beta = Y_mat.shape[1]
    alpha = cp.Variable(shape = (1,), name = 'alpha')
    beta = cp.Variable(shape = (dim_beta,), name = 'beta', nonneg=True)
    lambda_ = cp.Variable(shape = (1,), name = 'lambda', nonneg=True)
    inside_exp = cp.Variable(shape = (num_sample,), name = 'inside_exp')
    constraints = [
        cp.sum(beta) == 1,
        inside_exp >= -reg_params*(Y_mat@beta) + alpha + cp.quad_over_lin(reg_params*beta,4*lambda_),
        inside_exp >= (-(reg_params+1/tau)*(Y_mat@beta) +
                       (1-1/tau)*alpha + cp.quad_over_lin((reg_params+1/tau)*beta,4*lambda_)),
    ]
    problem = cp.Problem(cp.Minimize(lambda_*rho + cp.sum(inside_exp)/num_sample), constraints)
    problem.solve()
    if problem.status != 'optimal':
        raise ValueError('problem is not optimal')
    return beta.value

def cond_mean_CVaR_kernel(X_mat: np.array, Y_mat: np.array, X0: np.array, reg_params: float, tau: float, neighbor_quantile: float):
    X_mat = np.asarray(X_mat, dtype=np.float64)
    X0 = np.asarray(X0, dtype=np.float64).reshape(1, -1)
    Y_mat = np.asarray(Y_mat, dtype=np.float64)

    X_dist = np.linalg.norm(X_mat-X0, axis = 1)
    idx = (X_dist <= np.quantile(X_dist, neighbor_quantile))
    dim_beta = Y_mat.shape[1]
    dim_data = np.sum(idx)
    alpha = cp.Variable(shape = (1,), name = 'alpha')
    beta = cp.Variable(shape = (dim_beta,), name = 'beta', nonneg=True)
    lambda_ = cp.Variable(shape = (dim_data,), name = 'lambda')
    constraints = [
        cp.sum(beta) == 1,
        lambda_ >= -reg_params*(Y_mat[idx,:]@beta) + alpha,
        lambda_ >= -(reg_params+1/tau)*(Y_mat[idx,:]@beta) + (1-1/tau)*alpha,
    ]
    problem = cp.Problem(cp.Minimize(cp.sum(lambda_)), constraints)
    problem.solve()
    if problem.status != 'optimal':
        raise ValueError('problem is not optimal')
    return beta.value

def DR_Winf_conditional_mean_CVaR_kernel(X_mat: np.array, Y_mat: np.array, X0: np.array, reg_params: float, tau: float, gamma_quantile: float, rho_quantile: float):
    X_mat = np.asarray(X_mat, dtype=np.float64)
    X0 = np.asarray(X0, dtype=np.float64).reshape(1, -1)
    Y_mat = np.asarray(Y_mat, dtype=np.float64)

    eta = reg_params
    tau_inv = 1 / tau
    X_dist = np.linalg.norm(X_mat - X0, axis=1)
    X_dist[np.isnan(X_dist)] = 1e8
    gamma = np.quantile(X_dist, gamma_quantile)
    rho = np.quantile(X_dist, rho_quantile)
    idx_I = (X_dist <= gamma + rho)
    idx_I1 = (X_dist + rho <= gamma)
    idx_I2 = idx_I & (~idx_I1)
    norm_x_minus_xp_in_I = X_dist[idx_I] - gamma
    norm_x_minus_xp_in_I[norm_x_minus_xp_in_I < 0] = 0
    y_I = Y_mat[idx_I]

    stock_num = Y_mat.shape[1]
    beta = cp.Variable(stock_num, nonneg=True)
    alpha = cp.Variable(1)
    lambda_ = cp.Variable(shape=(1,))
    u = cp.Variable(shape=(len(y_I),), name='u')
    v_term_1 = alpha - eta * (Y_mat[idx_I] @ beta) + eta * cp.norm(beta) * (rho - norm_x_minus_xp_in_I)
    v_term_2 = ((1 - tau_inv) * alpha
                - (eta + tau_inv) * (Y_mat[idx_I] @ beta)
                + (eta + tau_inv) * cp.norm(beta) * (rho - norm_x_minus_xp_in_I))
    constraints = [
        u[idx_I2[idx_I]] >= 0,
        cp.sum(u) <= 0,
        cp.sum(beta) == 1,
        lambda_ + u >= v_term_1,
        lambda_ + u >= v_term_2
    ]
    problem = cp.Problem(cp.Minimize(lambda_), constraints)
    problem.solve(solver=cp.MOSEK, mosek_params={"MSK_IPAR_NUM_THREADS":1})
    if problem.status != 'optimal':
        raise ValueError('problem is not optimal')
    return beta.value

def DR_W2_conditional_mean_CVaR_kernel(X_mat: np.array, Y_mat: np.array, X0: np.array, reg_params: float, tau: float, epsilon: float, rho_div_rho_min: float,):
    X_mat = np.asarray(X_mat, dtype=np.float64)
    X0 = np.asarray(X0, dtype=np.float64).reshape(1, -1)
    Y_mat = np.asarray(Y_mat, dtype=np.float64)

    def compute_rho_min(X_mat, X0, epsilon):
        X_dist = np.linalg.norm(X_mat - X0, axis=1)
        X_dist[np.isnan(X_dist)] = 1e8
        X_cut = np.quantile(X_dist, q=epsilon, method='higher')
        return (X_dist[X_dist <= X_cut]**2).mean() * epsilon

    rho = rho_div_rho_min * compute_rho_min(X_mat, X0, epsilon)
    X_dist = np.linalg.norm(X_mat - X0, axis=1)
    eta = reg_params
    epsilon_inv = 1 / epsilon
    tau_inv = 1 / tau

    N, stock_num = Y_mat.shape
    beta = cp.Variable(stock_num, nonneg=True)
    alpha = cp.Variable(1)
    lambda1 = cp.Variable(1, nonneg=True)
    lambda2 = cp.Variable(1)
    theta = cp.Variable(N, nonneg=True)
    z = cp.Variable(N, nonneg=True)
    z_tilde = cp.Variable(N, nonneg=True)

    obj = cp.Minimize(lambda1 * rho + lambda2 * epsilon + cp.sum(theta) / N)
    linear_constraints = [
        cp.sum(beta) == 1,
        z == theta + lambda1 * X_dist ** 2 + lambda2 + epsilon_inv * eta * (Y_mat @ beta - alpha),
        z_tilde == (theta + lambda1 * X_dist ** 2 + lambda2
                    + epsilon_inv * (eta + tau_inv) * (Y_mat @ beta)
                    - epsilon_inv * (1 - tau_inv) * alpha)
    ]
    quad_over_lin_constraints = [
        z >= cp.quad_over_lin(epsilon_inv * eta * beta, 4 * lambda1),
        z_tilde >= cp.quad_over_lin(epsilon_inv * (eta + tau_inv) * beta, 4 * lambda1),
    ]
    problem = cp.Problem(obj, linear_constraints + quad_over_lin_constraints)
    problem.solve()
    if problem.status != 'optimal':
        raise ValueError('problem is not optimal')
    return beta.value

In [ ]:
def compute_avg_return(xi_mat):
    return np.mean(xi_mat, axis=1)

def NW_weights(x, X_train, h):
    dists = np.linalg.norm(X_train - x, axis=1)
    weights = np.exp(-0.5 * (dists / h)**2)
    return weights / np.sum(weights)

def LSCV_bandwidth(x_train, y_train, bandwidths):
    n = len(x_train)
    errors = []
    for h in bandwidths:
        total_error = 0
        for i in range(n):
            x_i = x_train[i]
            y_i = y_train[i]
            X_rest = np.delete(x_train, i, axis=0)
            y_rest = np.delete(y_train, i)
            w = NW_weights(x_i, X_rest, h)
            y_hat = np.sum(w * y_rest)
            total_error += (y_i - y_hat) ** 2
        errors.append(total_error)
    best_h = bandwidths[np.argmin(errors)]
    return best_h

def preprocess_side_info(s, xi, bandwidth_candidates=None, verbose=None, tag=""):
    if verbose is None:
        verbose = globals().get('BW_VERBOSE', True)

    s = np.asarray(s, dtype=float)
    if bandwidth_candidates is None:
        bandwidth_candidates = np.logspace(-2, 1, 20)

    y = compute_avg_return(xi)  # (T,)
    s_scaled = s.copy()
    h_list = []

    if verbose and tag:
        print(f"[preprocess_side_info:{tag}] selecting bandwidths...")

    for j in range(s.shape[1]):
        x_j = s[:, j].reshape(-1, 1)
        h_j = LSCV_bandwidth(x_j, y, bandwidth_candidates)
        h_list.append(h_j)
        s_scaled[:, j] = s[:, j] / h_j
        if verbose:
            print(f"Side info {j}: selected bandwidth h = {h_j:.4f}")

    return s_scaled, h_list

In [ ]:
def _ensure_time_col(df, prefer=("time","Time","date","Date","datetime","Datetime","timestamp","Timestamp")):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    if "time" not in df.columns:
        found = None
        for c in prefer:
            if c in df.columns:
                found = c
                break
        if found is None:
            found = df.columns[0]
        df = df.rename(columns={found: "time"})
    df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.normalize()
    df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)
    return df

def _upper_cols(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    cols = ["time"] + [c for c in df.columns if c != "time"]
    df = df[cols]
    df.columns = [("time" if c=="time" else c.upper()) for c in df.columns]
    return df

In [ ]:
# Main-Loop
SIDEINFO_CSV = "../data/clean_sideinfo.csv"
RETURNS_CSV  = "../data/clean_returns.csv"
SP500_MASK_CSV = "../data/sp500_membership_mask_on_returns_dates.csv" 

SIDE_COLS_CANON = ["CL=F","GSPC","DJI","TNX","VIX"]

# OOS window
OOS_START = "2022-01-01"
OOS_END   = "2023-12-31"

MIN_TRAIN_DAYS = 500 

# Hyperparameter grids
rho_list_DRMC = [0.05, 0.10, 0.5]
neighbor_quantile_list_CMC = [0.05, 0.10, 0.5]
gamma_quantile_list_DRCMC = [0.05, 0.10, 0.5]
rho_quantile_list_DRCMC   = [0.05, 0.10, 0.5]
quantile_level_list_OTCMC = [0.05, 0.10, 0.5]
rho_div_rho_min_list_OCTMC = [1.00, 1.50, 2.00]
eps_list_GMM = [0.01, 0.05, 0.1, 0.5, 1.0, 1.5, 2.0]

# Core loss parameters
tau = 0.1
eta_list = [1, 3, 5, 7, 9]

# GMM/NF training knobs
GMM_COVARIANCE_TYPE = "diag"
GMM_REG_COVAR = 1e-3
GMM_N_INIT = 10

max_K = 3
hidden_node = 64
hidden_layer = 2
block_size = 2
bins = 8
total_epoch = 500
device = "cpu"

PROGRESS = True        
BW_VERBOSE = True      
CV_VERBOSE = True

side_raw = pd.read_csv(SIDEINFO_CSV, low_memory=False)
side_raw = _ensure_time_col(side_raw)
side_raw = _upper_cols(side_raw)

side_rename = {"^GSPC":"GSPC", "^DJI":"DJI", "^TNX":"TNX", "^VIX":"VIX"}
side_raw = side_raw.rename(columns=side_rename)

missing_side = [c for c in SIDE_COLS_CANON if c not in side_raw.columns]
if missing_side:
    raise ValueError(
        f"Missing side columns in sideinfo CSV: {missing_side}. "
        f"Available cols (head): {list(side_raw.columns)[:30]}"
    )

side_df = side_raw[["time"] + SIDE_COLS_CANON].copy()
for c in SIDE_COLS_CANON:
    side_df[c] = pd.to_numeric(side_df[c], errors="coerce")

rets_raw = pd.read_csv(RETURNS_CSV, low_memory=False)
rets_raw = _ensure_time_col(rets_raw)
rets_raw = _upper_cols(rets_raw)

spm_raw = pd.read_csv(SP500_MASK_CSV, low_memory=False)
spm_raw = _ensure_time_col(spm_raw)
spm_raw = _upper_cols(spm_raw)

common_dates = pd.Index(side_df["time"]).intersection(rets_raw["time"]).intersection(spm_raw["time"])
side_df = side_df[side_df["time"].isin(common_dates)].copy()
rets_raw = rets_raw[rets_raw["time"].isin(common_dates)].copy()
spm_raw  = spm_raw[spm_raw["time"].isin(common_dates)].copy()

side_df = side_df.sort_values("time").reset_index(drop=True)
rets_raw = rets_raw.sort_values("time").reset_index(drop=True)
spm_raw  = spm_raw.sort_values("time").reset_index(drop=True)

ret_cols_all = [c for c in rets_raw.columns if c != "time"]
mask_cols_all = [c for c in spm_raw.columns if c != "time"]
common_tickers = sorted(list(set(ret_cols_all).intersection(mask_cols_all)))

if len(common_tickers) == 0:
    raise ValueError("No common tickers between clean_returns.csv and SP500 mask CSV.")

rets_raw = rets_raw[["time"] + common_tickers].copy()
spm_raw  = spm_raw[["time"] + common_tickers].copy()

data_scaled = side_df.merge(rets_raw, on="time", how="inner")
membership_mask = spm_raw.set_index("time").astype(int)  

asset_cols = common_tickers
dim_s = len(SIDE_COLS_CANON)
dim_xi = len(asset_cols)

print("Loaded data_scaled:", data_scaled.shape,
      "| dim_s=", dim_s, "| dim_xi=", dim_xi,
      "| range:", data_scaled["time"].min().date(), "to", data_scaled["time"].max().date())

def _portfolio_vec(x, tickers, label=""): 
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.shape[0] != len(tickers):
        raise ValueError(f"[{label}] portfolio dim mismatch: len(x)={x.shape[0]} vs d={len(tickers)}")
    return x        


def cvar_upper_tail(losses, tau=0.1):
    losses = np.asarray(losses, dtype=float).reshape(-1)
    T = len(losses)
    k = max(1, int(np.ceil(tau * T)))
    worst = np.partition(losses, -k)[-k:]
    return float(np.mean(worst))

def select_DRMC_multiday(X_train, Y_train, X_val_days, Y_val_days, rho_grid, eta, tau, idx_days, verbose=True):
    best_rho, best = None, float("inf")

    for rho in rho_grid:
        r_list = []
        for t in idx_days:
            X0 = X_val_days[t].reshape(1, -1)
            w  = DR_mean_CVaR_kernel(X_train, Y_train, X0, eta, tau, rho=rho)
            r_list.append(float(Y_val_days[t] @ w))  

        loss_list = [-r for r in r_list]
        cvar_loss = cvar_upper_tail(loss_list, tau=tau)
        mean_ret  = float(np.mean(r_list))
        score = cvar_loss - eta * mean_ret           

        if verbose:
            print(f"[DRMC] rho={rho:.4f} risk={score:.6g}")

        if score < best:
            best = score
            best_rho = rho

    return best_rho, best

def select_CMC_multiday(X_train, Y_train, X_val_days, Y_val_days, q_grid, eta, tau, idx_days, verbose=True):
    best_q, best = None, float("inf")
    for q in q_grid:
        r_list = []
        for t in idx_days:
            X0 = X_val_days[t].reshape(1, -1)
            w  = cond_mean_CVaR_kernel(X_train, Y_train, X0, eta, tau, neighbor_quantile=q)
            r_list.append(float(Y_val_days[t] @ w))

        loss_list = [-r for r in r_list]
        cvar_loss = cvar_upper_tail(loss_list, tau=tau)
        mean_ret  = float(np.mean(r_list))
        score = cvar_loss - eta * mean_ret

        if verbose:
            print(f"[CMC] q={q:.4f} risk={score:.6g}")

        if score < best:
            best = score
            best_q = q

    return best_q, best

def select_DRCMC_multiday(X_train, Y_train, X_val_days, Y_val_days, gamma_grid, rho_grid, eta, tau, idx_days, verbose=True):
    best_gamma, best_rho, best = None, None, float("inf")
    for g in gamma_grid:
        for rho in rho_grid:
            r_list = []
            for t in idx_days:
                X0 = X_val_days[t].reshape(1, -1)
                w  = DR_Winf_conditional_mean_CVaR_kernel(X_train, Y_train, X0, eta, tau, g, rho)
                r_list.append(float(Y_val_days[t] @ w))

            loss_list = [-r for r in r_list]
            cvar_loss = cvar_upper_tail(loss_list, tau=tau)
            mean_ret  = float(np.mean(r_list))
            score = cvar_loss - eta * mean_ret

            if verbose:
                print(f"[DRCMC] gamma={g:.4f} rho={rho:.4f} risk={score:.6g}")

            if score < best:
                best = score
                best_gamma, best_rho = g, rho

    return best_gamma, best_rho, best

def select_OTCMC_multiday(X_train, Y_train, X_val_days, Y_val_days, q_grid, rho_div_grid, eta, tau, idx_days, verbose=True):
    best_q, best_rho_div, best = None, None, float("inf")
    for q in q_grid:
        for rho_div in rho_div_grid:
            r_list = []
            for t in idx_days:
                X0 = X_val_days[t].reshape(1, -1)
                w  = DR_W2_conditional_mean_CVaR_kernel(X_train, Y_train, X0, eta, tau, q, rho_div)
                r_list.append(float(Y_val_days[t] @ w))

            loss_list = [-r for r in r_list]
            cvar_loss = cvar_upper_tail(loss_list, tau=tau)
            mean_ret  = float(np.mean(r_list))
            score = cvar_loss - eta * mean_ret

            if verbose:
                print(f"[OTCMC] q={q:.4f} rho_div={rho_div:.4f} risk={score:.6g}")

            if score < best:
                best = score
                best_q, best_rho_div = q, rho_div

    return best_q, best_rho_div, best

def quarter_starts_between(times, start, end):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    times = pd.to_datetime(times)
    mask = (times >= start) & (times <= end)
    idx = pd.DatetimeIndex(times[mask].unique()).sort_values()
    if len(idx) == 0:
        return []
    quarters = pd.period_range(idx.min().to_period("Q"), idx.max().to_period("Q"), freq="Q")
    out = []
    for q in quarters:
        q0 = q.start_time
        q1 = (q + 1).start_time
        in_quarter = idx[(idx >= q0) & (idx < q1)]
        if len(in_quarter) > 0:
            out.append(in_quarter[0])
    return out

def slice_by_time(df, t0, t1):
    m = (df["time"] >= t0) & (df["time"] < t1)
    return df.loc[m].copy()

def get_train_val_oos_slices_quarter(quarter_start_trading_day):
    qs = pd.to_datetime(quarter_start_trading_day)
    q_period = qs.to_period("Q")

    cal_qs = q_period.start_time
    cal_qnext = (q_period + 1).start_time

    val_start = cal_qs - pd.DateOffset(months=1)
    train_start = cal_qs - pd.DateOffset(years=2)

    train_df_full = slice_by_time(data_scaled, train_start, cal_qs)
    val_df        = slice_by_time(data_scaled, val_start, cal_qs)
    oos_df        = slice_by_time(data_scaled, qs, cal_qnext)

    return train_df_full, val_df, oos_df, train_start, val_start, cal_qnext

def sp500_members_on_date(d):
    d = pd.to_datetime(d).normalize()
    if d not in membership_mask.index:
        return []
    row = membership_mask.loc[d]
    return row.index[row == 1].tolist()

def eligible_assets_sp500_rule(train_df_full, train_df_cv, val_df, oos_df, quarter_start):
    ms = pd.to_datetime(quarter_start).normalize()
    sp = set(sp500_members_on_date(ms))
    if not sp:
        return []

    cand = [c for c in asset_cols if c in sp]

    def _complete_cols(df, cols):
        if df is None or len(df) == 0:
            return []
        x = df[cols]
        ok = (x.isna().sum(axis=0) == 0)
        return x.columns[ok].tolist()

    elig = set(_complete_cols(train_df_full, cand))   
    elig &= set(_complete_cols(train_df_cv, cand))    
    elig &= set(_complete_cols(val_df, cand))         
    elig &= set(_complete_cols(oos_df, cand))         
    return sorted(list(elig))


DAILY_ROWS = []

def _to_numpy(a): 
    if isinstance(a, torch.Tensor):
        return a.detach().cpu().numpy()
    return np.asarray(a)

def fit_gmm_nf_once(s_train_raw, xi_train_raw, fixed_K=None):
    s_train_raw  = np.asarray(s_train_raw, dtype=float)
    xi_train_raw = np.asarray(xi_train_raw, dtype=float)
    if s_train_raw.ndim != 2:
        raise ValueError(f"s_train_raw must be 2D, got shape {s_train_raw.shape}")
    if xi_train_raw.ndim != 2:
        raise ValueError(f"xi_train_raw must be 2D, got shape {xi_train_raw.shape}")
    if s_train_raw.shape[0] != xi_train_raw.shape[0]:
        raise ValueError(f"n mismatch: s rows={s_train_raw.shape[0]} vs xi rows={xi_train_raw.shape[0]}")

    d_local = int(xi_train_raw.shape[1])  
    dim_s_local = int(s_train_raw.shape[1])

    scaler_s = StandardScaler()
    scaler_xi = StandardScaler()

    s_train_std  = scaler_s.fit_transform(s_train_raw)
    xi_train_std = scaler_xi.fit_transform(xi_train_raw)

    joint_train_std = np.hstack([s_train_std, xi_train_std])  

    if fixed_K is None:
        best_K_local = int(select_K_by_AIC(joint_train_std, max_K=max_K, covariance_type=GMM_COVARIANCE_TYPE, reg_covar=GMM_REG_COVAR))
    else:
        best_K_local = int(fixed_K)

    base_gmm = GaussianMixture(n_components=best_K_local, covariance_type=GMM_COVARIANCE_TYPE, reg_covar=GMM_REG_COVAR,
                               n_init=GMM_N_INIT, random_state=42).fit(joint_train_std)

    joint_train_tensor = torch.tensor(joint_train_std, dtype=torch.float32, device=device)
    latent_size = int(joint_train_std.shape[1])

    nfm, loss_hist = train_nf_model(latent_size, best_K_local, hidden_node, hidden_layer, bins, block_size,
                                    total_epoch, joint_train_tensor, device, base_gmm=base_gmm)

    mu_base = base_gmm.means_
    p_base = base_gmm.weights_

    if base_gmm.covariance_type == "diag":
        sig_base = np.array([np.diag(base_gmm.covariances_[k]) for k in range(best_K_local)])
    elif base_gmm.covariance_type == "full":
        sig_base = np.array(base_gmm.covariances_)
    else:
        raise ValueError(f"Unsupported covariance_type: {base_gmm.covariance_type}")

    return {
        "nfm": nfm,
        "base_gmm": base_gmm,
        "K": best_K_local,
        "scaler_s": scaler_s,
        "scaler_xi": scaler_xi,
        "mu_base": mu_base,
        "sig_base": sig_base,
        "p_base": p_base,
        "dim_s": dim_s_local,
        "d": d_local,
        "latent_size": latent_size,
        "gmm_covariance_type": GMM_COVARIANCE_TYPE,
    }

def fit_nf_and_latent_gmm_once(*args, **kwargs):
    return fit_gmm_nf_once(*args, **kwargs)

def sample_xi_given_s(model, s0_raw, n_mc=100, seed=None):
    K = int(model["K"])
    nfm = model["nfm"]
    scaler_s = model["scaler_s"]
    scaler_xi = model["scaler_xi"]
    mu_base = model["mu_base"]
    sig_base = model["sig_base"]
    p_base = model["p_base"]
    dim_s_local = int(model["dim_s"])
    d_local = int(model["d"])

    s0_raw = np.asarray(s0_raw, dtype=float).reshape(1, -1)
    if s0_raw.shape[1] != dim_s_local:
        raise ValueError(f"s0_raw dim mismatch: got {s0_raw.shape[1]} but dim_s={dim_s_local}")

    s0_std = scaler_s.transform(s0_raw)
    xi_zero_std = np.zeros((1, d_local), dtype=float)
    x_aug_std = np.hstack([s0_std, xi_zero_std])

    x_aug_tensor = torch.tensor(x_aug_std, dtype=torch.float32, device=device)
    z_aug = _to_numpy(inverse(nfm, x_aug_tensor))[0]
    z_s = z_aug[:dim_s_local]

    mu_cond, cov_cond, p_cond = transforming_conditional(z_s, K, mu_base, sig_base, p_base, dim_s_local)
    z_xi_sample = MC_sampling(K, int(n_mc), mu_cond, cov_cond, p_cond, seed=seed)
    z_full = np.hstack([np.repeat(z_s.reshape(1, -1), int(n_mc), axis=0), z_xi_sample])

    z_tensor = torch.tensor(z_full, dtype=torch.float32, device=device)
    x_gen_std = _to_numpy(forward(nfm, z_tensor))
    xi_mc_std = x_gen_std[:, dim_s_local:]
    xi_mc = scaler_xi.inverse_transform(xi_mc_std)

    if xi_mc.shape != (int(n_mc), d_local):
        raise ValueError(f"sample_xi_given_s returned shape {xi_mc.shape}, expected {(int(n_mc), d_local)}")
    return xi_mc

def run_quarter(quarter_start, eta):
    train_df_full, val_df, oos_df, train_start, val_start, _quarter_end = get_train_val_oos_slices_quarter(quarter_start)

    def make_seed(dt, tag=0):
        base = int(pd.Timestamp(dt).strftime("%Y%m%d"))
        return (base * 100 + tag) % (2**32 - 1)
    

    train_df_cv = slice_by_time(data_scaled, train_start, val_start)

    quarter_label = str(pd.to_datetime(quarter_start).to_period("Q"))
    out = {"quarter": quarter_label, "train_days": int(len(train_df_full)), "cv_train_days": int(len(train_df_cv)), "val_days": int(len(val_df)), "oos_days": int(len(oos_df))}

    if len(train_df_full) < MIN_TRAIN_DAYS:
        out["SKIP"] = True
        out["reason"] = f"train_days<{MIN_TRAIN_DAYS}"
        out["d"] = 0
        return out
    if len(train_df_cv) == 0 or len(val_df) == 0 or len(oos_df) == 0:
        out["SKIP"] = True
        out["reason"] = "empty cv_train/val/oos slice"
        out["d"] = 0
        return out

    elig = eligible_assets_sp500_rule(train_df_full, train_df_cv, val_df, oos_df, quarter_start)
    out["d"] = int(len(elig))
    if len(elig) == 0:
        out["SKIP"] = True
        out["reason"] = "no eligible assets"
        return out

    s_train_cv    = train_df_cv[SIDE_COLS_CANON].values
    xi_train_cv   = train_df_cv[elig].values
    s_train_full  = train_df_full[SIDE_COLS_CANON].values
    xi_train_full = train_df_full[elig].values

    s_train_full_scaled, h_list_full = preprocess_side_info(s_train_full, xi_train_full, verbose=BW_VERBOSE)
    h_full = np.asarray(h_list_full, dtype=float).reshape(1, -1)

    s_train_cv_scaled, h_list_cv = preprocess_side_info(s_train_cv, xi_train_cv, verbose=BW_VERBOSE)
    h_cv = np.asarray(h_list_cv, dtype=float).reshape(1, -1)

    s_val_days_raw = val_df[SIDE_COLS_CANON].values          
    xi_val_days    = val_df[elig].values                 
    Tval = int(len(val_df))
    if Tval == 0:
        out["SKIP"] = True
        out["reason"] = "val_df is empty (cannot CV kernel params)"
        return out

    idx_days = np.arange(Tval, dtype=int)
    out["val_days"] = int(Tval)
    out["cv_days_used"] = int(len(idx_days))
    assert len(idx_days) == Tval
    X_val_days_scaled = s_val_days_raw / h_cv               

    best_rho_DRMC, val_loss_DRMC = select_DRMC_multiday(s_train_cv_scaled, xi_train_cv, X_val_days_scaled, xi_val_days, rho_list_DRMC, eta, tau, idx_days, verbose=BW_VERBOSE)
    best_q_CMC, val_loss_CMC = select_CMC_multiday(s_train_cv_scaled, xi_train_cv, X_val_days_scaled, xi_val_days, neighbor_quantile_list_CMC, eta, tau, idx_days, verbose=BW_VERBOSE)
    best_gamma, best_rho_DRCMC, val_loss_DRCMC = select_DRCMC_multiday(s_train_cv_scaled, xi_train_cv, X_val_days_scaled, xi_val_days, gamma_quantile_list_DRCMC, rho_quantile_list_DRCMC, eta, tau, idx_days, verbose=BW_VERBOSE)
    best_q_OT, best_rho_div, val_loss_OTCMC = select_OTCMC_multiday(s_train_cv_scaled, xi_train_cv, X_val_days_scaled, xi_val_days, quantile_level_list_OTCMC, rho_div_rho_min_list_OCTMC, eta, tau, idx_days, verbose=BW_VERBOSE)

    out.update({"best_rho_DRMC": best_rho_DRMC,"best_q_CMC": best_q_CMC,"best_gamma_DRCMC": best_gamma,"best_rho_DRCMC": best_rho_DRCMC,"best_q_OTCMC": best_q_OT,"best_rho_div_OTCMC": best_rho_div,
                "val_loss_DRMC": float(val_loss_DRMC),"val_loss_CMC": float(val_loss_CMC),"val_loss_DRCMC": float(val_loss_DRCMC),"val_loss_OTCMC": float(val_loss_OTCMC)})

    out["GMM_K"] = np.nan
    out["GMM_eps"] = np.nan
    out["GMM_covariance_type"] = None

    scaler_s_tmp = StandardScaler()
    scaler_xi_tmp = StandardScaler()
    s_cv_std_tmp  = scaler_s_tmp.fit_transform(s_train_cv)
    xi_cv_std_tmp = scaler_xi_tmp.fit_transform(xi_train_cv)
    joint_cv_std_tmp = np.hstack([s_cv_std_tmp, xi_cv_std_tmp])

    K_quarter = int(select_K_by_AIC(joint_cv_std_tmp, max_K=max_K, covariance_type=GMM_COVARIANCE_TYPE, reg_covar=GMM_REG_COVAR))
    out["GMM_K"] = K_quarter

    try:
        if PROGRESS:
            print(f"[{out['quarter']}] Fitting base GMM + NF...", flush=True)

        with torch.enable_grad():
            model_cv   = fit_gmm_nf_once(s_train_cv,   xi_train_cv, fixed_K=K_quarter)
            model_full = fit_gmm_nf_once(s_train_full, xi_train_full, fixed_K=K_quarter)

        try:
            model_full["nfm"].eval()
        except Exception:
            pass

        assert model_full["mu_base"].shape[1] == model_full["dim_s"] + model_full["d"]
        assert len(model_full["sig_base"]) == model_full["K"]
        out["GMM_covariance_type"] = model_full["gmm_covariance_type"]
        out["GMM_K"] = K_quarter

        if PROGRESS:
            print(f"[{out['quarter']}] Base GMM + NF fit done. K={out['GMM_K']}, covariance={out['GMM_covariance_type']}", flush=True)

    except Exception as e:
        out["SKIP"] = True
        out["reason"] = f"Base GMM + NF fit failed: {repr(e)}"
        print(f"[{out['quarter']}] SKIP: {out['reason']}", flush=True)
        return out

    best_eps, best_score = None, float("inf")
    try:
        if PROGRESS:
            print(f"[{out['quarter']}] GMM eps-CV start...", flush=True)

        s_val_days  = val_df[SIDE_COLS_CANON].values
        xi_val_days = val_df[elig].values
        Tval = int(len(val_df))
        if Tval == 0:
            raise ValueError("val_df is empty; cannot CV eps.")

        xi_mc_cache = []
        with torch.no_grad():
            for t in idx_days:
                s0 = s_val_days[t].reshape(1, -1)
                seed_t = make_seed(val_df["time"].iloc[t], tag=1)
                xi_mc_cache.append(sample_xi_given_s(model_cv, s0, n_mc=100, seed=seed_t))

        for eps in eps_list_GMM:
            r_list = []   
            for j, t in enumerate(idx_days):
                w = Portfolio_2_Wass_MCVaR(xi_mc_cache[j], eps, tau, eta)
                w = _portfolio_vec(w, elig, label="GMM_CV")
                r_list.append(float(xi_val_days[t] @ w))  

            loss_list = [-r for r in r_list]
            cvar_loss = cvar_upper_tail(loss_list, tau=tau)
            mean_ret  = float(np.mean(r_list))
            score = cvar_loss - eta * mean_ret          

            if CV_VERBOSE:
                print(f"[Base GMM + NF CV] quarter={out['quarter']} eps={eps:.4f} risk={score:.6g}", flush=True)

            if score < best_score:
                best_score = score
                best_eps = eps

        if best_eps is None:
            raise RuntimeError("best_eps stayed None (unexpected). Check eps_list_GMM or CV loss.")

        out["GMM_eps"] = float(best_eps)

        if PROGRESS:
            print(f"[{out['quarter']}] GMM eps-CV done. best_eps={best_eps} best_risk={best_score:.6g}", flush=True)

    except Exception as e:
        out["SKIP"] = True
        out["reason"] = f"eps CV failed: {repr(e)}"
        print(f"[{out['quarter']}] SKIP: {out['reason']}", flush=True)
        return out

    times_oos   = pd.to_datetime(oos_df["time"]).values
    s_oos_days  = oos_df[SIDE_COLS_CANON].values
    xi_oos_days = oos_df[elig].values
    n_days = len(times_oos)
    if n_days == 0:
        out["SKIP"] = False
        return out

    def _process_one_day(i_day: int):
        t_day = times_oos[i_day]
        row = {"time": t_day, "quarter": out["quarter"], "month": pd.to_datetime(t_day).strftime("%Y-%m"), "d": int(len(elig))}
        s0_raw  = s_oos_days[i_day].reshape(1, -1)              
        xi_true = np.asarray(xi_oos_days[i_day], dtype=float).reshape(-1)

        try:
            with torch.no_grad():
                seed_day = make_seed(t_day, tag=2)
                xi_mc_today = sample_xi_given_s(model_full, s0_raw, n_mc=100, seed=seed_day)
        except Exception as e:
            for mname in ["EW","MC","DRMC","CMC","DRCMC","OTCMC","GMM"]:
                row[f"ret_{mname}"]  = np.nan
            row["GMM_K"] = float(out["GMM_K"])
            row["GMM_eps"] = float(out["GMM_eps"])
            row["GMM_day_error"] = f"xi_mc sampling failed: {repr(e)}"
            return i_day, row

        X0_full = (s0_raw / h_full).reshape(1, -1)   
        X0_cv   = (s0_raw / h_cv).reshape(1, -1)     
        X_train_full_fullmetric = s_train_full_scaled         
        X_train_full_cvmetric   = (s_train_full / h_cv)        

        w_EW = _portfolio_vec(equal_weight_kernel(X_train_full_fullmetric, xi_train_full, X0_full), elig, "EW_daily")
        w_MC = _portfolio_vec(mean_CVaR_kernel(X_train_full_fullmetric, xi_train_full, X0_full, eta, tau), elig, "MC_daily")
        w_DRMC = _portfolio_vec(DR_mean_CVaR_kernel(X_train_full_fullmetric, xi_train_full, X0_full, eta, tau, rho=best_rho_DRMC), elig, "DRMC_daily")
        w_CMC = _portfolio_vec(cond_mean_CVaR_kernel(X_train_full_cvmetric, xi_train_full, X0_cv, eta, tau, neighbor_quantile=best_q_CMC), elig, "CMC_daily")
        w_DRCMC = _portfolio_vec(DR_Winf_conditional_mean_CVaR_kernel(X_train_full_cvmetric, xi_train_full, X0_cv, eta, tau, best_gamma, best_rho_DRCMC), elig, "DRCMC_daily")
        w_OT = _portfolio_vec(DR_W2_conditional_mean_CVaR_kernel(X_train_full_cvmetric, xi_train_full, X0_cv, eta, tau, best_q_OT, best_rho_div), elig, "OTCMC_daily")

        packs_today = {"EW": w_EW, "MC": w_MC, "DRMC": w_DRMC, "CMC": w_CMC, "DRCMC": w_DRCMC, "OTCMC": w_OT}

        for mname, wvec in packs_today.items():
            r = float(xi_true @ wvec)
            row[f"ret_{mname}"]  = r

        try:
            w_gmm = Portfolio_2_Wass_MCVaR(xi_mc_today, float(out["GMM_eps"]), tau, eta)
            w_gmm = _portfolio_vec(w_gmm, elig, "GMM_daily")
            r_gmm = float(xi_true @ w_gmm)
            row["ret_GMM"]  = r_gmm
            row["GMM_K"] = float(out["GMM_K"])
            row["GMM_eps"] = float(out["GMM_eps"])

        except Exception as e:
            row["ret_GMM"] = np.nan
            row["GMM_K"] = float(out["GMM_K"])
            row["GMM_eps"] = float(out["GMM_eps"])
            row["GMM_day_error"] = repr(e)

        if PROGRESS:
            print(f"[DAILY] {pd.to_datetime(t_day).date()}, ret: {r_gmm}", flush=True)

        return i_day, row

    rows_local = [None] * n_days

    max_workers = min(12, os.cpu_count() or 1, n_days)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_process_one_day, i) for i in range(n_days)]
        for fut in as_completed(futs):
            i_day, row = fut.result()
            rows_local[i_day] = row

    DAILY_ROWS.extend(rows_local)

    out["SKIP"] = False
    return out

# RUN
quarter_starts = quarter_starts_between(data_scaled["time"].values, OOS_START, OOS_END)
print("Quarter starts:", len(quarter_starts), quarter_starts[:3], "...")

out_dir = Path("Results")
out_dir.mkdir(parents=True, exist_ok=True)

for eta in eta_list:
    print("\n" + "=" * 80)
    print(f"RUNNING eta = {eta}")
    print("=" * 80)

    DAILY_ROWS = []
    for q0 in quarter_starts:
        if PROGRESS:
            print(f"\n[{pd.to_datetime(q0).to_period('Q')}] running...")
        run_quarter(q0, eta)

    daily_df = pd.DataFrame(DAILY_ROWS)
    if len(daily_df) > 0:
        daily_df["time"] = pd.to_datetime(daily_df["time"])
        daily_df = daily_df.sort_values(["time", "quarter", "month"]).reset_index(drop=True)

    daily_path = out_dir / f"PF_eta{int(eta)}.csv"
    daily_df.to_csv(daily_path, index=False)
    print("Saved daily results to:", daily_path)